# ShopNhanh — Project 01: Basket & Sales EDA cho ShopNhanh Mart

**Học viên:** ____________ &nbsp;&nbsp; **Ngày nộp:** ____________

Đọc `business_brief.md` và `data_dictionary.md` trước khi bắt đầu. Bạn là Data Analyst của team Category ở **ShopNhanh**. Câu hỏi bạn cần trả lời: *Q4 này nên cấu hình 5 combo "Thường được mua cùng" nào để thu thêm nhiều lợi nhuận gộp nhất?*

| # | Section | Marks |
|---|---|---|
| 1 | Understanding the Problem | 10 |
| 2 | Setup & Data Loading | 3 |
| 3 | Data Quality Check | 10 |
| 4 | Cleaning & Joining | 12 |
| 5 | Sales EDA | 12 |
| 6 | Market Basket Analysis | 18 |
| 7 | Time & Segment Slicing | 10 |
| 8 | Business Impact | 12 |
| 9 | Monitoring | 3 |
| E | Extension | 10 |
| | **Total** | **100** |

**Quy ước:**
- Mỗi phần có một ô `# Your code here` và một ô **Commentary**.
- Commentary phải là **kết luận** kèm con số ("… nên team Category nên …"), không chỉ mô tả lại biểu đồ.
- Bạn được phép (và nên) **quay lại Section 4** khi phát hiện thêm vấn đề dữ liệu ở Section 5–6. Dữ liệu thật luôn đi theo vòng lặp như vậy.
- Thư mục `.solution/` chứa lời giải mẫu. **Đừng mở trước khi nộp bài:** phần tự khám phá dữ liệu là phần bạn học được nhiều nhất.

## 1. Understanding the Problem (10 marks)

Trả lời ngắn gọn, 1–3 câu cho mỗi ý:
1. Mục tiêu của team Category trong **một câu**, có đơn vị tiền.
2. Đây là loại bài toán gì (supervised hay unsupervised)? Trong dữ liệu này, **một giỏ hàng** được tạo từ bảng nào và lọc ra sao?
3. Vì sao chọn 5 combo có **lift cao nhất**, hoặc 5 cặp **hay được mua cùng nhất**, đều chưa phải cách đúng? Đại lượng nào mới quyết định?
4. Ràng buộc "chỉ 5 slot, mỗi A một combo, mỗi cặp một lần" ảnh hưởng thế nào đến cách bạn xếp hạng combo?

*Your answer here.*

## 2. Setup & Data Loading (3 marks)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations

RANDOM_STATE = 42
DATA_DIR = 'data'
SCALE, MONTHS, TAKE_K = 20, 3, 0.20     # từ business brief: mẫu 5%, 3 tháng, take_rate = 0.20 × confidence
MIN_SUPPORT, MIN_LIFT = 0.005, 1.2      # ràng buộc cho mỗi combo

pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid')

In [ ]:
# Load 4 bảng: customers, orders, order_items, products.
# Gợi ý: pd.read_csv(f'{DATA_DIR}/orders.csv', parse_dates=['order_datetime'])
# In shape và head() của từng bảng.

# Your code here

## 3. Data Quality Check (10 marks)

Ở phần này bạn chỉ **phát hiện** vấn đề, chưa sửa. Sửa ở Section 4. Kết quả cuối cùng của phần này là **bảng vấn đề dữ liệu** ở mục 3.5.

### 3.1 Kiểu dữ liệu & giá trị thiếu

In [ ]:
# Gợi ý: df.info(), df.isna().mean().sort_values(ascending=False)

# Your code here

**Commentary:** *Your answer here.*

### 3.2 Dòng trùng lặp

In [ ]:
# Gợi ý: df.duplicated().sum(); orders['order_id'].is_unique

# Your code here

**Commentary:** *Your answer here.*

### 3.3 Toàn vẹn khoá ngoại (referential integrity)

In [ ]:
# Mỗi khoá ngoại có trỏ tới một dòng tồn tại không?
# Gợi ý: (~orders['customer_id'].isin(customers['customer_id'])).sum()
# Kiểm tra: orders → customers, order_items → orders, order_items → products

# Your code here

**Commentary:** *Your answer here.*

### 3.4 Giá trị bất thường

In [ ]:
# Gợi ý:
#   order_items[['quantity', 'unit_price']].describe()
#   orders['status'].value_counts(normalize=True)
#   products.sort_values('list_price').head()
#   order_items['bundle_id'].value_counts(dropna=False)
#   orders['customer_id'].value_counts().head()        # có khách nào đặt nhiều bất thường?

# Your code here

**Commentary:** *Your answer here.*

### 3.5 Bảng vấn đề dữ liệu

| # | Vấn đề | Bảng | Số dòng ảnh hưởng | Cách xử lý ở Section 4 (và lý do) |
|---|---|---|---|---|
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |
| 4 | | | | |
| 5 | | | | |
| 6 | | | | |

## 4. Cleaning & Joining (12 marks)

Kết quả cần có:
- `lines`: 1 dòng = 1 dòng hàng *sạch*, có thêm cột từ `products`, `orders` và `customers`, cùng `revenue = quantity × unit_price` và `gross_profit = revenue × gross_margin`.
- `baskets`: 1 dòng = 1 **đơn completed**, gồm `n_products`, `n_units`, `revenue`, `gross_profit`, `order_datetime`, `segment`, `city`.
- `cleaning_log`: bảng ghi số dòng trước và sau mỗi bước làm sạch.

In [ ]:
cleaning_log = []

def log(step, before, after):
    """Ghi lại số dòng trước/sau mỗi bước làm sạch."""
    cleaning_log.append({'step': step, 'rows_before': before, 'rows_after': after, 'removed': before - after})

### 4.1 Làm sạch

In [ ]:
# Xử lý từng vấn đề trong bảng 3.5. Mỗi bước gọi log(...).
# Ví dụ:
#   before = len(order_items)
#   order_items = order_items.drop_duplicates()
#   log('order_items: drop duplicates', before, len(order_items))

# Your code here

### 4.2 Join → `lines`

In [ ]:
# Gợi ý: order_items.merge(products, on='product_id', how='left', validate='many_to_one')
#        validate='many_to_one' sẽ báo lỗi nếu bảng bên phải bị trùng khoá: một cách kiểm tra rẻ mà hiệu quả.

# Your code here

### 4.3 Aggregate → `baskets`

In [ ]:
# Gợi ý: lines.groupby('order_id').agg(n_products=('product_id', 'nunique'), revenue=('revenue', 'sum'), ...)
#        Sau đó join thông tin cấp đơn (order_datetime, segment, city).
# In pd.DataFrame(cleaning_log)

# Your code here

**Commentary:** *Your answer here.*

## 5. Sales EDA (12 marks)

Mỗi biểu đồ cần có tiêu đề, nhãn trục và đơn vị. Commentary trả lời câu hỏi *"điều này nghĩa là gì với team Category?"*

### 5.1 Doanh thu theo tuần

In [ ]:
# Gợi ý: baskets.set_index('order_datetime').resample('W')['revenue'].sum().plot()

# Your code here

**Commentary:** *Your answer here.*

### 5.2 Category: doanh thu so với lợi nhuận gộp

In [ ]:
# Category nào chiếm nhiều doanh thu nhưng mang lại ít lời, và ngược lại?

# Your code here

**Commentary:** *Your answer here.*

### 5.3 Phân phối số sản phẩm/đơn và AOV

In [ ]:
# Histogram của n_products và revenue. Có giỏ hàng nào bất thường không?

# Your code here

**Commentary:** *Your answer here.*

### 5.4 Segment & thành phố

In [ ]:
# AOV, n_products trung bình, top category của từng segment.

# Your code here

**Commentary:** *Your answer here.*

### 5.5 Ngày trong tuần × giờ

In [ ]:
# Gợi ý: pivot = baskets.pivot_table(index=baskets.order_datetime.dt.dayofweek,
#                                    columns=baskets.order_datetime.dt.hour,
#                                    values='order_id', aggfunc='count')   # cần cột order_id
#        sns.heatmap(pivot)

# Your code here

**Commentary:** *Your answer here.*

## 6. Market Basket Analysis (18 marks)

Đây là cách đếm của lab `1_market_basket_analysis.ipynb`, áp dụng cho dữ liệu thật. Mọi tỷ lệ chia cho **N = số giỏ hàng** (số đơn completed sạch), không phải số dòng hàng.

| Metric | Công thức | Ý nghĩa |
|---|---|---|
| **Support(A, B)** | `n_ab / N` | Cặp này phổ biến cỡ nào? |
| **Confidence(A → B)** | `n_ab / n_a` | Trong các giỏ có A, bao nhiêu % có B? |
| **Lift(A → B)** | `confidence / (n_b / N)` | Mạnh hơn ngẫu nhiên bao nhiêu lần? (> 1 = có liên hệ) |

### 6.1 Support của từng sản phẩm

In [ ]:
# item_count = số giỏ chứa mỗi sản phẩm. Vẽ top 15.

# Your code here

**Commentary:** *Your answer here.*

### 6.2 Đếm cặp sản phẩm

In [ ]:
# Cách 1 (giống lab): dict + itertools.combinations trên từng giỏ hàng.
# Cách 2 (pandas, nhanh hơn):
#   basket_items = lines[['order_id', 'product_id']].drop_duplicates()
#   pairs = basket_items.merge(basket_items, on='order_id', suffixes=('_a', '_b'))
#   pairs = pairs[pairs.product_id_a != pairs.product_id_b]     # giữ cả hai chiều A→B và B→A

# Your code here

### 6.3 Tính rules

In [ ]:
# Viết hàm để dùng lại ở Section 7:
# def mine_rules(lines_subset):
#     """Trả về DataFrame: product_a, product_b, n_ab, n_a, n_b, support, confidence, lift."""
#     ...
# rules = mine_rules(lines)

# Your code here

### 6.4 Top rules

In [ ]:
# Lọc support >= MIN_SUPPORT và lift >= MIN_LIFT, thêm tên sản phẩm, xem top 15 theo lift.

# Your code here

**Commentary:** *Your answer here.*

### 6.5 Sanity check

Trong top rules ở 6.4, rule nào **không phải** combo tốt để đưa lên widget? Với mỗi rule đó, nêu lý do kèm bằng chứng từ dữ liệu. Nếu tìm ra vấn đề mới, quay lại Section 4, sửa, rồi chạy lại các ô phía dưới.

*Your answer here.*

## 7. Time & Segment Slicing (10 marks)

Một combo có thể chỉ mạnh vào một lúc nhất định hoặc với một nhóm khách nhất định. Khi gộp cả quý, tín hiệu đó bị loãng.

### 7.1 Feature thời gian

In [ ]:
# Thêm vào lines/baskets: weekday (0 = thứ 2), hour, month,
# is_weekend_evening = (thứ 6–CN) & (giờ >= 18)

# Your code here

### 7.2 Rules trên từng lát cắt

In [ ]:
# Chạy lại mine_rules() trên ít nhất 3 lát cắt và so sánh support/confidence/lift của cùng một rule, ví dụ:
#   - tháng 9 so với tháng 7–8
#   - is_weekend_evening == True so với False
#   - một segment so với phần còn lại
# Lưu ý: N của mỗi lát cắt là số giỏ hàng trong lát cắt đó.

# Your code here

### 7.3 Commentary

Combo nào **chỉ hiện ra** (hoặc mạnh hơn hẳn) khi xét trong một lát cắt? Có nên dùng combo đó cho Q4 không? Nếu có thì nên dùng theo cách nào?

*Your answer here.*

## 8. Business Impact (12 marks)

```
value_month = SCALE × (n_a − n_ab) / MONTHS × TAKE_K × confidence(A→B) × avg_price_B × gross_margin_B
```

### 8.1 `value_month` cho mọi rule ứng viên

In [ ]:
# avg_price_B = đơn giá thực bán trung bình của B trong lines

# Your code here

### 8.2 Chọn 5 combo

In [ ]:
# Áp dụng đủ các ràng buộc trong brief. Kết quả là DataFrame combo_recommendations, 5 dòng.
# combo_recommendations.to_csv('combo_recommendations.csv', index=False)

# Your code here

### 8.3 So sánh với cách làm "ngây thơ"

In [ ]:
# Bảng tổng value_month của 3 cách chọn 5 combo:
#   (a) của bạn  (b) top-5 theo lift  (c) top-5 theo n_ab (hay mua cùng nhất)
# Combo vi phạm ràng buộc trong brief tính value = 0.

# Your code here

**Commentary:** *Your answer here.*

### 8.4 Khuyến nghị cho team Category

Viết 5–7 câu gửi Trưởng team Category, gồm:
- 5 combo bạn chọn;
- tổng lợi nhuận gộp tăng thêm mỗi tháng và mỗi quý;
- giả định lớn nhất trong con số đó;
- 1 đề xuất ngoài 5 slot.

*Your answer here.*

## 9. Monitoring (3 marks)

1. Sau khi bật 5 combo, bạn theo dõi những chỉ số nào mỗi tuần, cho từng combo?
2. Làm sao biết lợi nhuận tăng thêm là **do widget**, chứ không phải do mùa vụ hay do một chương trình khuyến mãi khác? *(Gợi ý: Project 03 sẽ dạy cách này.)*
3. Khi nào cần chạy lại phân tích để thay combo?

*Your answer here.*

## E. Extension (10 marks: 2 × 5)

### E1. Combo 3 sản phẩm (5 marks)

In [ ]:
# Rule {A, B} → C với support >= 0.3%. Top 5 theo lift. Có rule 3 sản phẩm nào đáng giá hơn rule 2 sản phẩm không?

# Your code here

**Commentary:** *Your answer here.*

### E2. Hàm `recommend(product_id, k=3)` (5 marks)

In [ ]:
# Trả về top-k sản phẩm B nên gợi ý khi giỏ có product_id, xếp theo value_month, thoả ràng buộc support/lift.
# Thử với P001, P027, P043.

# Your code here

**Commentary:** *Your answer here.*